# 01 - 冯·诺依曼架构（Von Neumann Architecture）

这个 notebook 用 **Python 模拟** 带你亲手体验计算机的核心架构。你将：
- 用代码构建一台「纸面计算机」
- 理解指令和数据共存于内存是怎样的
- 亲眼看到取指-译码-执行循环

## 怎么用？
从上到下依次运行每个 cell（`Shift + Enter`），观察输出，自己动手改参数实验。

## 今日学习路线

| # | 内容 |
|---|------|
| 1.1 | 冯·诺依曼架构简介 |
| 1.2 | 用 Python 模拟存储程序概念 |
| 1.3 | 五大组件模拟 |
| 1.4 | 取指-译码-执行全流程 |
| 1.5 | 冯·诺依曼瓶颈可视化 |
| 1.6 | 哈佛架构对比 |

---
## 1.1 冯·诺依曼架构简介

### 一句话总结

**程序和指令以相同的二进制形式存放在同一个存储器里。** 这听起来理所当然，但在 1945 年，这是革命性的。

### 五大组件

```
输入设备 ──→ 内存(Memory) ──→ 输出设备
               ↑↓
       CPU (控制器CU + 运算器ALU + 寄存器)
```

所有组件通过**三大总线**通信：
- **数据总线**：传数据（内存 ↔ CPU）
- **地址总线**：传地址（CPU → 内存，告诉它要读/写哪里）
- **控制总线**：传信号（读、写、中断……）

---
## 1.2 内存模型：指令和数据共存

先来看最简单的例子：内存里混着放指令和数据。

In [ ]:
# ============================================
# 内存模拟：就是一个大列表
# 每个单元可以存指令或数据（都是数字）
# ============================================

class Memory:
    """模拟计算机内存"""
    def __init__(self, size=256):
        self.size = size
        self.cells = [0] * size  # 全部初始化为 0
    
    def read(self, address):
        """从指定地址读取数据"""
        if 0 <= address < self.size:
            return self.cells[address]
        raise ValueError(f"地址 {address} 越界！")
    
    def write(self, address, value):
        """向指定地址写入数据"""
        if 0 <= address < self.size:
            self.cells[address] = value
        else:
            raise ValueError(f"地址 {address} 越界！")
    
    def dump(self, start, end):
        """打印内存区域的快照"""
        print(f"{'地址':>6s} | {'值':>6s} | {'二进制':>10s} | 说明")
        print('-' * 55)
        for addr in range(start, min(end, self.size)):
            val = self.cells[addr]
            binary = f'{val:08b}' if val >= 0 else f'??'
            print(f"{addr:6d} | {val:6d} | {binary:>10s}")

# 创建内存
mem = Memory(256)
print(f"✅ 创建了 {mem.size} 字节的内存，全部初始化为 0\n")
print("前 8 个地址的快照：")
mem.dump(0, 8)

In [ ]:
# ============================================
# 在内存里同时放入「指令」和「数据」
# 这就是存储程序的核心：它们长得一样
# ============================================

# 我们定义一套简单的指令编码（操作码）：
# 1 = LOAD  (加载数据到 ACC)
# 2 = ADD   (ACC + 内存值 → ACC)
# 3 = STORE (ACC → 内存)
# 4 = PRINT (打印 ACC)
# 5 = HALT  (停机)
# 99= 数据标记（不是指令）

# 程序：计算 10 + 20，结果存到地址 50
mem.write(0, 1)   # 地址0: LOAD
mem.write(1, 10)  # 地址1: 操作数 = 地址10（那里存着数字 10）
mem.write(2, 2)   # 地址2: ADD
mem.write(3, 11)  # 地址3: 操作数 = 地址11（那里存着数字 20）
mem.write(4, 3)   # 地址4: STORE
mem.write(5, 50)  # 地址5: 操作数 = 地址50（存结果）
mem.write(6, 4)   # 地址6: PRINT
mem.write(7, 5)   # 地址7: HALT

# 数据区
mem.write(10, 10)  # 地址10: 数据 = 10
mem.write(11, 20)  # 地址11: 数据 = 20

print("📋 内存快照 — 指令和数据混在一起\n")
print(f"{'地址':>6s} | {'值':>6s} | 含义")
print('-' * 35)

labels = {
    0: 'LOAD (操作码=1)', 1: '→ 加载地址 10 的值',
    2: 'ADD  (操作码=2)', 3: '→ 加上地址 11 的值',
    4: 'STORE(操作码=3)', 5: '→ 存到地址 50',
    6: 'PRINT(操作码=4)', 7: '→ HALT(操作码=5)',
    10: '数据: 10', 11: '数据: 20'
}
for addr in range(13):
    val = mem.read(addr)
    label = labels.get(addr, '')
    print(f"  {addr:4d} | {val:6d} | {label}")

### ✏️ 观察

仔细看上面的内存——**指令（1, 2, 3, 4, 5）和数据（10, 20）混在同一个列表里**。CPU 靠 **PC（程序计数器）** 知道「现在该读的是指令还是数据」。

这就是冯·诺依曼架构的核心：**程序即数据**。这意味着：
- 程序可以修改自身（自修改代码）
- 编译器可以把源代码翻译成数据（指令），放进内存就能跑
- 操作系统切换任务时，只要保存/恢复内存就行

---
## 1.3 五大组件模拟

下面用 Python 逐一模拟冯·诺依曼的五大部分。

In [ ]:
# ============================================
# 完整模拟：五大部分
# ============================================

class SimpleComputer:
    """一台简单的冯·诺依曼计算机"""
    
    def __init__(self, memory_size=256):
        # ---- 存储器 (Memory) ----
        self.memory = Memory(memory_size)
        
        # ---- CPU 内部寄存器 ----
        self.PC = 0    # Program Counter: 指向下一条指令的地址
        self.IR = 0    # Instruction Register: 存放当前指令
        self.ACC = 0   # Accumulator: 累加器，存放运算结果
        self.MAR = 0   # Memory Address Register: 存放要访问的内存地址
        self.MBR = 0   # Memory Buffer Register: 存放从内存读/写的数据
        
        # ---- 状态标志 ----
        self.running = False
        self.step_count = 0
        
        # ---- 输入输出缓冲 ----
        self.output_buffer = []
    
    def load_program(self, instructions, data=None):
        """加载程序和数据到内存（输入设备 → 内存）"""
        # 写入指令
        for addr, val in instructions.items():
            self.memory.write(addr, val)
        # 写入数据
        if data:
            for addr, val in data.items():
                self.memory.write(addr, val)
        print(f"✅ 加载了 {len(instructions)} 条指令")
    
    def step(self):
        """执行一条指令的完整周期：取指 → 译码 → 执行"""
        
        # ========== FETCH（取指）==========
        # 1. 把 PC 的值送到地址总线 (MAR = PC)
        self.MAR = self.PC
        # 2. 从内存读取指令 (通过数据总线到 MBR)
        self.MBR = self.memory.read(self.MAR)
        # 3. 指令送入 IR
        self.IR = self.MBR
        # 4. PC + 1，指向下一条指令或操作数
        self.PC += 1
        
        opcode = self.IR
        self.step_count += 1
        
        # ========== DECODE & EXECUTE（译码 + 执行）==========
        if opcode == 1:  # LOAD
            # 读操作数地址
            self.MAR = self.memory.read(self.PC)
            self.PC += 1
            # 从该地址加载数据到 ACC
            self.MBR = self.memory.read(self.MAR)
            self.ACC = self.MBR
            print(f"  Step {self.step_count}: LOAD  [地址{self.MAR}] → ACC = {self.ACC}")
            
        elif opcode == 2:  # ADD
            self.MAR = self.memory.read(self.PC)
            self.PC += 1
            self.MBR = self.memory.read(self.MAR)
            old_acc = self.ACC
            self.ACC += self.MBR
            print(f"  Step {self.step_count}: ADD   [地址{self.MAR}] {old_acc} + {self.MBR} = {self.ACC}")
            
        elif opcode == 3:  # STORE
            self.MAR = self.memory.read(self.PC)
            self.PC += 1
            self.memory.write(self.MAR, self.ACC)
            print(f"  Step {self.step_count}: STORE ACC={self.ACC} → [地址{self.MAR}]")
            
        elif opcode == 4:  # PRINT
            msg = f"🖨️  输出: {self.ACC}"
            self.output_buffer.append(self.ACC)
            print(f"  Step {self.step_count}: PRINT {msg}")
            
        elif opcode == 5:  # HALT
            self.running = False
            print(f"  Step {self.step_count}: HALT ⏹️  停机")
            
        else:
            print(f"  Step {self.step_count}: ❌ 未知指令 {opcode}")
            self.running = False
    
    def run(self):
        """运行程序直到 HALT"""
        self.running = True
        self.PC = 0
        self.step_count = 0
        print("=" * 50)
        print("🚀 开始执行...")
        print("=" * 50)
        
        while self.running and self.PC < self.memory.size:
            self.step()
            if self.step_count > 100:  # 防止死循环
                print("⚠️ 超过 100 步，强制停止")
                break
        
        print("=" * 50)
        print(f"✅ 执行完毕，共 {self.step_count} 步")
        return self.output_buffer

# 创建计算机实例
computer = SimpleComputer()
print("✅ 计算机已初始化")
print(f"   内存大小: {computer.memory.size} 字节")
print(f"   寄存器: PC, IR, ACC, MAR, MBR")

In [ ]:
# ============================================
# 加载我们之前写的程序并执行
# ============================================

computer = SimpleComputer()

# 程序：计算 10 + 25，打印结果
program = {
    0: 1,   # LOAD
    1: 10,  #   从地址10加载
    2: 2,   # ADD
    3: 11,  #   加上地址11的值
    4: 4,   # PRINT
    5: 5,   # HALT
}

data = {
    10: 10,  # 第一个数
    11: 25,  # 第二个数
}

computer.load_program(program, data)
computer.run()

### ✏️ 自己动手

修改下面 `program` 和 `data` 字典，写一个新程序：**计算 100 - 30**（提示：先定义一个 SUB 操作码 = 6，你需要在 `SimpleComputer.step()` 里加入 SUB 的实现，或者用 ADD 加上一个负数）。

更简单的方式：直接在 data 里放负数！

In [ ]:
# 👇 修改下面的 program 和 data，实现 100 + (-30) = 70
computer2 = SimpleComputer()

my_program = {
    0: 1,   # LOAD
    1: 10,  #   从地址10加载
    2: 2,   # ADD
    3: 11,  #   加上地址11的值
    4: 4,   # PRINT
    5: 5,   # HALT
}

my_data = {
    10: 100,  # 👈 改这里
    11: -30,  # 👈 改这里（负数等于减法）
}

computer2.load_program(my_program, my_data)
computer2.run()

---
## 1.4 取指-译码-执行全流程追踪

下面我们用一张「状态表」追踪 CPU 在每个步骤中所有寄存器的变化，让你看清数据在五大组件间如何流动。

In [ ]:
import pandas as pd

class TraceComputer(SimpleComputer):
    """带执行追踪的计算机"""
    def __init__(self, memory_size=256):
        super().__init__(memory_size)
        self.trace = []  # 记录每一步的寄存器状态
    
    def step(self):
        # 记录步骤前状态
        before = {
            '步骤': self.step_count + 1,
            'PC(改前)': self.PC,
            'MAR': self.MAR,
            'MBR': self.MBR,
            'IR(指令)': self.IR,
            'ACC(累加器)': self.ACC,
        }
        
        super().step()
        
        # 记录步骤后状态
        after = {
            'PC(改后)': self.PC,
            'ACC(改后)': self.ACC,
        }
        self.trace.append({**before, **after})

# 运行追踪版本
tracer = TraceComputer()

program = {
    0: 1, 1: 10,   # LOAD [10]
    2: 2, 3: 11,   # ADD  [11]
    4: 3, 5: 20,   # STORE [20]
    6: 1, 7: 20,   # LOAD [20]（验证存储是否成功）
    8: 4,           # PRINT
    9: 5,           # HALT
}
data = {10: 50, 11: 7}

tracer.load_program(program, data)
tracer.run()

# 用表格展示追踪结果
print("\n📊 寄存器状态变化追踪表：\n")
df = pd.DataFrame(tracer.trace)
print(df.to_string(index=False))

### 追踪表解读

每一行是一条指令的完整执行。注意观察：
1. **PC** 如何递增（指向下一条指令）
2. **IR** 当前执行的指令编码
3. **ACC** 数据如何在累加器中流动
4. **MAR/MBR** 如何充当 CPU 和内存之间的「中间人」

---
## 1.5 冯·诺依曼瓶颈可视化

CPU 处理指令的速度远快于从内存读取指令/数据的速度——CPU 经常要「等」内存。这就是**冯·诺依曼瓶颈**。

下面我们模拟一下这个现象。

In [ ]:
import time
import random

def simulate_cpu_vs_memory():
    """
    模拟 CPU 运算速度和内存访问速度的差异
    """
    # 模拟参数（数量级近似）
    cpu_cycle_ns = 0.3      # CPU 一个周期 ~0.3 ns (3GHz)
    l1_cache_ns = 0.5        # L1 缓存访问 ~0.5 ns
    l2_cache_ns = 7          # L2 缓存访问 ~7 ns
    ram_ns = 100             # 内存访问 ~100 ns  ← 瓶颈！
    
    operations = 1000
    
    print("📊 执行 1000 次操作的模拟耗时：\n")
    print(f"{'存储层级':<20s} | {'每次访问耗时':>15s} | {'总耗时':>15s} | {'相对CPU':>10s}")
    print("-" * 75)
    
    for name, ns in [("CPU 周期", cpu_cycle_ns),
                      ("L1 Cache", l1_cache_ns),
                      ("L2 Cache", l2_cache_ns),
                      ("主存 RAM", ram_ns)]:
        total = ns * operations
        ratio = ns / cpu_cycle_ns
        print(f"{name:<20s} | {ns:>10.1f} ns    | {total:>10.1f} ns    | {ratio:>7.0f}x")
    
    print("\n" + "=" * 60)
    print("🔑 关键结论：")
    print(f"   访问一次 RAM 的时间 ≈ CPU 执行 {(ram_ns / cpu_cycle_ns):.0f} 个周期")
    print(f"   如果程序经常访问内存，CPU 大部分时间在「等」")
    print(f"   → 这就是为什么需要 Cache！（详见第 4 章）")

simulate_cpu_vs_memory()

In [ ]:
# ============================================
# 可视化：CPU 等待内存的时间占比
# ============================================

def von_neumann_bottleneck_demo():
    """
    一个程序混合了计算指令和内存访问指令
    看看 CPU 有多少时间在等内存
    """
    # 假设：
    # - 1/3 的指令需要访问内存（LOAD/STORE）
    # - CPU 执行一条指令：1 个周期（0.3ns）
    # - 内存访问：额外等 100ns
    
    total_instructions = 100
    cpu_cycles_per_instr = 1
    mem_access_ratio = 0.33  # 33% 的指令访存
    
    cpu_time = total_instructions * cpu_cycles_per_instr * 0.3  # ns
    mem_wait_time = total_instructions * mem_access_ratio * 100  # ns
    total_time = cpu_time + mem_wait_time
    
    print("🕐 执行 100 条指令的时间分解：\n")
    print(f"  CPU 实际运算时间: {cpu_time:.0f} ns")
    print(f"  等待内存时间:     {mem_wait_time:.0f} ns")
    print(f"  总时间:           {total_time:.0f} ns")
    print(f"\n  🔴 CPU 等待内存的占比: {mem_wait_time / total_time * 100:.1f}%")
    print(f"  🟢 CPU 实际工作的占比: {cpu_time / total_time * 100:.1f}%")
    
    # 简单柱状图
    print("\n  可视化（每个字符 ≈ 1%）：")
    cpu_pct = int(cpu_time / total_time * 50)
    mem_pct = 50 - cpu_pct
    print(f"  CPU 工作: {'█' * max(1, cpu_pct)} {cpu_time / total_time * 100:.0f}%")
    print(f"  等待内存: {'█' * mem_pct} {mem_wait_time / total_time * 100:.0f}%")

von_neumann_bottleneck_demo()

### 💡 思考

如果 CPU 90% 以上的时间都在等内存，那提升 CPU 主频有什么意义呢？

这就是为什么现代计算机在 CPU 和内存之间插入了**多层缓存（Cache）**——把常用数据放在离 CPU 更近、更快的地方。详见第 4 章。

---
## 1.6 哈佛架构对比

哈佛架构的核心区别：**指令和数据有各自独立的存储空间和总线**。这意味着可以同时取指令和取数据。

下面模拟一下两种架构在执行同一段程序时的差异。

In [ ]:
# ============================================
# 冯·诺依曼 vs 哈佛架构模拟
# ============================================

class VonNeumannCPU:
    """冯·诺依曼架构：指令和数据共享总线"""
    def run_instruction(self, need_data=True):
        cycles = 1  # CPU 执行周期
        # 取指令（占用总线）
        cycles += 6  # 内存访问延迟
        if need_data:
            # 取数据（又要占用同一条总线，必须等上一次访存结束）
            cycles += 6
        return cycles

class HarvardCPU:
    """哈佛架构：指令和数据分开的总线"""
    def run_instruction(self, need_data=True):
        cycles = 1  # CPU 执行周期
        # 取指令和取数据可以重叠（两条总线同时工作）
        if need_data:
            # 两条总线并行，取总延迟是 max(6, 6) = 6，而非 6 + 6
            cycles += 6
        else:
            cycles += 6
        return cycles

# 模拟执行一段程序：100 条指令，其中 60% 需要访问数据
INSTR_COUNT = 100
MEM_ACCESS_RATIO = 0.6
mem_access_count = int(INSTR_COUNT * MEM_ACCESS_RATIO)
no_mem_count = INSTR_COUNT - mem_access_count

vn = VonNeumannCPU()
harvard = HarvardCPU()

vn_cycles = sum(vn.run_instruction(True) for _ in range(mem_access_count)) + \
            sum(vn.run_instruction(False) for _ in range(no_mem_count))

harvard_cycles = sum(harvard.run_instruction(True) for _ in range(mem_access_count)) + \
                 sum(harvard.run_instruction(False) for _ in range(no_mem_count))

print("📊 执行同一条程序（100 条指令，60% 访存）：\n")
print(f"  冯·诺依曼: {vn_cycles} 个周期")
print(f"  哈佛架构:   {harvard_cycles} 个周期")
print(f"  哈佛快了:   {(vn_cycles / harvard_cycles - 1) * 100:.0f}%")
print(f"\n💡 原因：哈佛架构可以在取指令的同时取数据（两条独立总线并行工作）")

### 两级架构的融合：改进的哈佛架构

现代 CPU 实际采用混合方案：

```
         ┌──────────────────┐
         │     CPU 核心      │
         │                  │
         │  ┌──────┐ ┌────┐ │
  L1 I$──┼──┤取指单元│ │执行│─┼──L1 D$
  (指令)  │  └──────┘ │单元│ │  (数据)
  Cache   │           └────┘ │   Cache
         └──────┬───────────┘
                │ 共享总线
         ┌──────▼───────────┐
         │    L2/L3 Cache   │
         │    + 主存 RAM     │  ← 对外是冯·诺依曼（统一存储）
         └──────────────────┘
```

- **片内**（L1 Cache）：指令 Cache 和数据 Cache 分开 → 哈佛方式
- **片外**（L2/L3/RAM）：统一编址，共享总线 → 冯·诺依曼方式

这样既获得了哈佛架构的并行优势，又保留了冯·诺依曼的灵活性。

---
## 🎯 综合挑战

用你的 `SimpleComputer` 写一段完整的程序。

In [ ]:
# Q1: 编写程序，计算 三个数的和：(地址10) + (地址11) + (地址12)
# 提示：LOAD 第一个数，ADD 第二个数，ADD 第三个数，PRINT，HALT

c = SimpleComputer()

prog = {
    # 👇 在这里写你的程序
}

d = {
    10: 5,
    11: 15,
    12: 25,
}

# c.load_program(prog, d)
# c.run()
print("✏️ 写完后取消上面两行的注释来运行")

In [ ]:
# Q2: 冯·诺依曼架构中，以下哪些是共享同一条总线的？
# A) 指令和数据
# B) 输入和输出
# C) 地址和数据
# D) 寄存器和 ALU

answer_q2 = ""  # 👈 填你的答案（如 "A"）
print(f"你的答案: {answer_q2}")

In [ ]:
# Q3: 如果一个系统有 32 条地址线，最多可以访问多少内存？
# 提示：n 条地址线 → 2^n 个地址单元，每个地址通常对应 1 字节

address_lines = 32
max_memory_bytes = 2 ** address_lines  # 👈 计算
max_memory_gb = max_memory_bytes / (1024 ** 3)
print(f"最大可寻址内存: {max_memory_bytes:,} 字节 = {max_memory_gb:.0f} GB")

---
## ✅ 检查清单

对照一下，都掌握了吗？

- [ ] 冯·诺依曼架构的五大组件：输入、输出、内存、控制器、ALU
- [ ] 存储程序概念：指令和数据以二进制形式共存于同一内存
- [ ] 系统总线的三种类型：数据总线、地址总线、控制总线
- [ ] 指令周期的四个阶段：取指→译码→执行→写回
- [ ] 冯·诺依曼瓶颈是什么，为什么会发生
- [ ] 冯·诺依曼 vs 哈佛架构的区别和应用场景
- [ ] 现代 CPU 如何混合两种架构（改进的哈佛架构）

---

> 📖 下一章：[02 - CPU 工作原理](../02-cpu-working-principle/) — 深入取指-译码-执行，理解 CPU 内部如何运转